# Kaggle API Server Runner

This notebook runs the persistent OpenAI-compatible server script.


In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import json
import os
import shutil
import signal
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from typing import Any, Dict, Optional

MODEL_ID = "Deign86/deped-math-qwen2.5-7b-deped-math-merged"
DISALLOWED_MODEL_IDS = {
    "Deign86/deped-math-qwen2.5-7b-checkpoint-700-lora",
    "Qwen/Qwen2.5-7B-Instruct",
}

HOST = "0.0.0.0"
PORT = int(os.getenv("API_PORT", "8000"))
DTYPE = os.getenv("VLLM_DTYPE", "float16").strip() or "float16"
MAX_MODEL_LEN = int(os.getenv("VLLM_MAX_MODEL_LEN", "2048"))
GPU_MEMORY_UTILIZATION = float(os.getenv("VLLM_GPU_MEMORY_UTILIZATION", "0.82"))
TENSOR_PARALLEL_SIZE = int(os.getenv("VLLM_TENSOR_PARALLEL_SIZE", "1"))
MAX_NUM_SEQS = int(os.getenv("VLLM_MAX_NUM_SEQS", "2"))
CPU_OFFLOAD_GB = float(os.getenv("VLLM_CPU_OFFLOAD_GB", "10"))
ATTENTION_BACKEND = os.getenv("VLLM_ATTENTION_BACKEND", "TRITON_ATTN").strip() or "TRITON_ATTN"
ENFORCE_EAGER = os.getenv("VLLM_ENFORCE_EAGER", "1").strip().lower() in {"1", "true", "yes"}
TRUST_REMOTE_CODE = os.getenv("VLLM_TRUST_REMOTE_CODE", "0").strip().lower() in {"1", "true", "yes"}

WORK_DIR = Path("/kaggle/working")
MODEL_LOCAL_DIR = WORK_DIR / "model"
RUNTIME_STATUS_PATH = WORK_DIR / "api_runtime_status.json"
CLOUDFLARED_LOG_PATH = WORK_DIR / "cloudflared.log"


def stage(title: str) -> None:
    print(f"\n{'=' * 12} {title} {'=' * 12}")


def run_cli(command: list[str], check: bool = True, env: Optional[Dict[str, str]] = None) -> subprocess.CompletedProcess[str]:
    cmd_env = os.environ.copy()
    if env:
        cmd_env.update(env)

    print("[cli]", " ".join(command))
    proc = subprocess.run(
        command,
        check=False,
        text=True,
        capture_output=True,
        env=cmd_env,
    )

    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())

    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(command)}")
    return proc


def pip_install(*packages: str) -> None:
    run_cli([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *packages], check=True)


def assert_model_lock() -> None:
    env_model_id = os.getenv("MODEL_ID", "").strip()
    if env_model_id and env_model_id != MODEL_ID:
        raise RuntimeError(
            "MODEL_ID override is not allowed for this kernel. "
            f"Expected {MODEL_ID}, got {env_model_id}."
        )
    if MODEL_ID in DISALLOWED_MODEL_IDS:
        raise RuntimeError(f"Configured MODEL_ID is disallowed: {MODEL_ID}")


def configure_hf_auth() -> Dict[str, str]:
    token = os.getenv("HF_TOKEN", "").strip()
    if not token:
        return {"used_token": "false", "source": "missing"}

    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token

    try:
        from huggingface_hub import login

        login(token=token, add_to_git_credential=False)
        return {"used_token": "true", "source": "HF_TOKEN", "method": "huggingface_hub.login"}
    except Exception as exc:
        print(f"HF login warning: {exc}")
        return {"used_token": "true", "source": "HF_TOKEN", "method": "env_only", "warning": str(exc)}


def _extract_special_token(item: object) -> str:
    if isinstance(item, str):
        return item.strip()
    if isinstance(item, dict):
        for key in ["token", "content", "value", "text"]:
            value = item.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()
    return ""


def prepare_model_artifacts() -> Path:
    from huggingface_hub import snapshot_download

    MODEL_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=MODEL_ID, local_dir=str(MODEL_LOCAL_DIR))

    tokenizer_config_path = MODEL_LOCAL_DIR / "tokenizer_config.json"
    if not tokenizer_config_path.exists():
        return MODEL_LOCAL_DIR

    try:
        tokenizer_config = json.loads(tokenizer_config_path.read_text(encoding="utf-8"))
    except Exception:
        return MODEL_LOCAL_DIR

    changed = False
    extra_special_tokens = tokenizer_config.get("extra_special_tokens")

    if isinstance(extra_special_tokens, list):
        extracted_tokens = [
            token
            for token in (_extract_special_token(item) for item in extra_special_tokens)
            if token
        ]
        additional_tokens = tokenizer_config.get("additional_special_tokens", [])
        if not isinstance(additional_tokens, list):
            additional_tokens = []
        merged = list(dict.fromkeys([*additional_tokens, *extracted_tokens]))
        tokenizer_config["additional_special_tokens"] = merged
        tokenizer_config["extra_special_tokens"] = {}
        changed = True
    elif extra_special_tokens is not None and not isinstance(extra_special_tokens, dict):
        tokenizer_config["extra_special_tokens"] = {}
        changed = True

    if changed:
        tokenizer_config_path.write_text(json.dumps(tokenizer_config, indent=2), encoding="utf-8")

    return MODEL_LOCAL_DIR


def ensure_cloudflared() -> Path:
    cloudflared_bin = shutil.which("cloudflared")
    if cloudflared_bin:
        return Path(cloudflared_bin)

    target = WORK_DIR / "cloudflared"
    if target.exists():
        target.chmod(0o755)
        return target

    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    print(f"Downloading cloudflared from {url}")
    urllib.request.urlretrieve(url, str(target))
    target.chmod(0o755)
    return target


def build_cloudflared_command(cloudflared_bin: Path) -> list[str]:
    return [
        str(cloudflared_bin),
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
        "--logfile",
        str(CLOUDFLARED_LOG_PATH),
        "--loglevel",
        "info",
    ]


def wait_for_local_api(vllm_proc: subprocess.Popen[Any], timeout_seconds: int = 600) -> None:
    import requests

    endpoint = f"http://127.0.0.1:{PORT}/v1/models"
    started = time.time()
    last_error = ""

    while time.time() - started < timeout_seconds:
        if vllm_proc.poll() is not None:
            raise RuntimeError("vLLM process exited before API became ready.")
        try:
            response = requests.get(endpoint, timeout=10)
            if response.status_code < 500:
                print(f"Local API reachable at {endpoint} (status={response.status_code}).")
                return
            last_error = f"status={response.status_code}"
        except Exception as exc:
            last_error = str(exc)
        time.sleep(3)

    raise RuntimeError(f"Timed out waiting for local vLLM API readiness: {last_error}")


def write_runtime_status(payload: Dict[str, object]) -> None:
    RUNTIME_STATUS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def terminate_process(proc: Optional[subprocess.Popen[Any]], name: str) -> None:
    if proc is None or proc.poll() is not None:
        return
    print(f"Stopping {name}...")
    proc.terminate()
    try:
        proc.wait(timeout=20)
    except Exception:
        proc.kill()


def main() -> None:
    stage("Model Lock")
    assert_model_lock()
    print(f"MODEL_ID locked to: {MODEL_ID}")

    stage("Install Dependencies")
    pip_install("--upgrade", "pip")
    pip_install("vllm>=0.6.0", "huggingface_hub>=0.23.0", "requests>=2.31.0")
    pip_install("protobuf>=4.25.3,<6")

    stage("Hugging Face Auth")
    hf_auth = configure_hf_auth()
    print(json.dumps(hf_auth, indent=2))

    stage("Prepare Model Artifacts")
    model_path = prepare_model_artifacts()
    print(f"MODEL_PATH: {model_path}")

    stage("Launch vLLM OpenAI API")
    vllm_command = [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        str(model_path),
        "--tokenizer",
        str(model_path),
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--dtype",
        DTYPE,
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--tensor-parallel-size",
        str(TENSOR_PARALLEL_SIZE),
        "--max-num-seqs",
        str(MAX_NUM_SEQS),
        "--cpu-offload-gb",
        str(CPU_OFFLOAD_GB),
        "--download-dir",
        "/kaggle/working/hf-cache",
        "--attention-backend",
        ATTENTION_BACKEND,
    ]
    if ENFORCE_EAGER:
        vllm_command.append("--enforce-eager")
    if TRUST_REMOTE_CODE:
        vllm_command.append("--trust-remote-code")

    vllm_proc = subprocess.Popen(vllm_command, text=True)

    cloudflared_proc: Optional[subprocess.Popen[Any]] = None

    def handle_signal(signum: int, _frame: object) -> None:
        print(f"Received signal {signum}; shutting down.")
        terminate_process(cloudflared_proc, "cloudflared")
        terminate_process(vllm_proc, "vLLM")
        raise SystemExit(0)

    signal.signal(signal.SIGTERM, handle_signal)
    signal.signal(signal.SIGINT, handle_signal)

    try:
        wait_for_local_api(vllm_proc=vllm_proc, timeout_seconds=900)

        stage("Launch Cloudflared Tunnel")
        cloudflared_bin = ensure_cloudflared()
        cloudflared_command = build_cloudflared_command(cloudflared_bin)
        if CLOUDFLARED_LOG_PATH.exists():
            CLOUDFLARED_LOG_PATH.unlink()

        cloudflared_proc = subprocess.Popen(cloudflared_command, text=True)

        time.sleep(5)
        if cloudflared_proc.poll() is not None:
            print("cloudflared exited right after startup; auto-restart will retry.")
            cloudflared_proc = None

        runtime_payload: Dict[str, object] = {
            "success": True,
            "model_id": MODEL_ID,
            "host": HOST,
            "port": PORT,
            "dtype": DTYPE,
            "max_model_len": MAX_MODEL_LEN,
            "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
            "max_num_seqs": MAX_NUM_SEQS,
            "cpu_offload_gb": CPU_OFFLOAD_GB,
            "attention_backend": ATTENTION_BACKEND,
            "enforce_eager": ENFORCE_EAGER,
        }
        write_runtime_status(runtime_payload)

        print("SERVER_READY: true")
        print("TUNNEL_URL_NOTE: Automatic URL extraction is disabled. Provide the tunnel URL manually from logs.")
        print(f"CLOUDFLARED_LOG_PATH: {CLOUDFLARED_LOG_PATH}")
        print(f"MODEL_ID: {MODEL_ID}")
        print(json.dumps(runtime_payload, indent=2))

        stage("Keep Server Running")
        while True:
            if vllm_proc.poll() is not None:
                print("vLLM exited; restarting vLLM and refreshing tunnel.")
                try:
                    vllm_proc = subprocess.Popen(vllm_command, text=True)
                    wait_for_local_api(vllm_proc=vllm_proc, timeout_seconds=900)
                    terminate_process(cloudflared_proc, "cloudflared")
                    cloudflared_proc = None
                    print("vLLM restart complete.")
                except Exception as restart_exc:
                    print(f"vLLM restart failed: {restart_exc}")
                    time.sleep(10)
                    continue

            if cloudflared_proc is None or cloudflared_proc.poll() is not None:
                print("cloudflared not running; attempting restart.")
                try:
                    if CLOUDFLARED_LOG_PATH.exists():
                        CLOUDFLARED_LOG_PATH.unlink()
                    cloudflared_proc = subprocess.Popen(cloudflared_command, text=True)
                    time.sleep(3)
                    if cloudflared_proc.poll() is not None:
                        print("cloudflared restart failed; will retry.")
                        cloudflared_proc = None
                        time.sleep(10)
                        continue
                    print("cloudflared restart complete.")
                except Exception as tunnel_exc:
                    print(f"cloudflared restart exception: {tunnel_exc}")
                    cloudflared_proc = None
                    time.sleep(10)
                    continue
            time.sleep(15)

    except Exception as exc:
        error_payload: Dict[str, object] = {
            "success": False,
            "model_id": MODEL_ID,
            "error": str(exc),
        }
        write_runtime_status(error_payload)
        print(json.dumps(error_payload, indent=2))
        terminate_process(cloudflared_proc, "cloudflared")
        terminate_process(vllm_proc, "vLLM")
        raise


if __name__ == "__main__":
    main()
